##CH 09. 추천 시스템
### 01. 추천 시스템의 개요와 배경
-추천 시스템의 개요
  -추천 시스템: 사용자의 취향을 이해하고 맞춤 상품과 콘텐츠를 제공하여 자신의 사이트에 고객이 머무르는 시간을 오래 소요하게 하는 것

- 추천 시스템의 장점은 매출을 큰 폭으로 올릴 수 있고, 사용자의 쇼핑 즐거움이 배가됨

- 추천 시스템의 핵심은 사용자 자신도 좋아하는지 몰랐던 취향을 시스템이 발견하고 그에 맞는 콘텐츠를 추천해주는 것

#### 온라인 스토어의 필수 요소, 추천 시스템
- 추천 시스템은 특히 온라인에서 그 진가를 발휘함
- 추천 엔진은 사용자가 무엇을 원하는지 빠르게 찾아내는 장점이 있어 사용자의 온라인 쇼핑 이용 즐거움을 배가 시킴

### 추천 시스템의 유형
추천 시스템의 2가지 유형
- 콘텐츠 기반 필터링
- 협업 필터링


### 02. 콘텐츠 기반 필터링 추천 시스템
콘텐츠 기반 필터링 방식: 특정 아이템을 매우 선호하는 경우, 그 아이템과 비슷한 콘텐츠를 가진 다른 아이템을 추천하는 방식

### 03. 초근접 이웃 협업 필터링

- 협업 필터링의 주요 목표

사용자-아이템 평점 매트릭스와 같은 축적된 사용자 행동 데이터를 기반으로 한 사용자가 아직 평가하지 않은 아이템을 예측 평가하는 것


-헙업 필터링 기반의 추천 시스템의 2가지 방식 : 최근접 이웃 방식, 잠재 요인 방식으로 나뉨

- 최근접 이웃 협업 필터링 : 사용자 기반과 아이템 기반으로 나뉨
  - 사용자 기반: 당신과 비슷한 고객들이 다음 상품도 구매했다는 취향이 비슷한 사람의 이력을 추천함

  - 아이템 기반: 이 상품을 선택한 다른 고객들은 다음 상품도 구매했다는 상품을 구매한 사람들의 이력을 추천함


### 04. 잠재요인 협업 필터링
#### 잠재 요인 협업 필터링의 이해
잠재 요인 협업 필터링: 사용자-아이템 평점 매트릭스 속에 숨어 있는 잠재 요인을 추출하여 추천 예측을 할 수 있게 하는 기법

#### 행렬 분해의 이해
- 행렬 분해: 다차원의 매트릭스를 저차원 매트릭스로 분해하는 기법으로 대표적으로 SVD, NMF 등이 있음
예시
- 행렬 분해는 보통은 SVD 방식을 이용하나, SVD는 결측값이 없는 행렬에만 적용할 수 있음

#### 확률적 경사하강법을 이용한 행렬 분해
- 핵심 로직: 비용함수를 최소화하는 방향성을 가지고 회귀 계수의 업데이트 값을 구한 뒤, 이 업데이트 값을 회귀 계수에 반복적으로 적용

In [1]:
import numpy as np

# 원본 행렬 R 생성, 분해 행렬 P와 Q 초기화, 잠재 요인 차원 K는 3으로 설정.
R = np.array([[4, np.nan, np.nan, 2, np.nan],
             [np.nan, 5, np.nan, 3, 1],
             [np.nan, np.nan, 3, 4, 4],
             [5, 2, 1, 2, np.nan]])
num_users, num_items = R.shape
K=3

# P와 Q 행렬의 크기를 지정하고 정규 분포를 가진 임의의 값으로 입력합니다.
np.random.seed(1)
P = np.random.normal(scale=1./K, size=(num_users, K))
Q = np.random.normal(scale=1./K, size=(num_items, K))

In [2]:
from sklearn.metrics import mean_squared_error

def get_rmse(R,P,Q,non_zeros):
  error=0
  # 두 개의 분해된 행렬 P와 Q.T의 내적으로 예측 R 행렬 생성
  full_pred_metrix = np.dot(P,Q.T)

  # 실제 R 행렬에서 널이 아닌 값의 위치 인덱스를 추출해 실제 R 행렬과 예측 행렬의 RMSE 추출
  x_non_zero_ind = [non_zero[0] for non_zero in non_zeros]
  y_non_zero_ind = [non_zero[1] for non_zero in non_zeros]
  R_non_zeros = R[x_non_zero_ind, y_non_zero_ind]
  full_pred_matrix_non_zeros = full_pred_metrix[x_non_zero_ind, y_non_zero_ind]
  mse = mean_squared_error(R_non_zeros, full_pred_matrix_non_zeros)
  rmse = np.sqrt(mse)

  return rmse

In [3]:
# R > 0인 행 위치, 열 위치, 값을 non_zeros 리스트에 저장.
non_zeros = [(i,j,R[i,j]) for i in range(num_users) for j in range(num_items) if R[i,j]>0]

steps=1000
learning_rate=0.01
r_lambda=0.01

# SGD 기법으로 P와 Q 매트릭스를 계속 업데이트.
for step in range(steps):
  for i,j,r in non_zeros:
    # 실제 값과 예측 값의 차이인 오류 값 구함
    eij = r- np.dot(P[i, :], Q[j,:].T)
    # Regularization을 반영한 SGD 업데이트 공식 적용
    P[i,:] = P[i,:] + learning_rate*(eij * Q[j,:]- r_lambda*P[i,:])
    Q[j,:] = Q[j,:] + learning_rate*(eij * P[i,:]- r_lambda*Q[j,:])

  rmse = get_rmse(R,P,Q, non_zeros)
  if (step %50) ==0:
    print("### iteration step:", step, "rmse:", rmse)

### iteration step: 0 rmse: 3.2388050277987723
### iteration step: 50 rmse: 0.4876723101369648
### iteration step: 100 rmse: 0.1564340384819247
### iteration step: 150 rmse: 0.07455141311978046
### iteration step: 200 rmse: 0.04325226798579314
### iteration step: 250 rmse: 0.029248328780878973
### iteration step: 300 rmse: 0.022621116143829466
### iteration step: 350 rmse: 0.019493636196525135
### iteration step: 400 rmse: 0.018022719092132704
### iteration step: 450 rmse: 0.01731968595344266
### iteration step: 500 rmse: 0.016973657887570753
### iteration step: 550 rmse: 0.016796804595895633
### iteration step: 600 rmse: 0.01670132290188466
### iteration step: 650 rmse: 0.01664473691247669
### iteration step: 700 rmse: 0.016605910068210026
### iteration step: 750 rmse: 0.016574200475705
### iteration step: 800 rmse: 0.01654431582921597
### iteration step: 850 rmse: 0.01651375177473524
### iteration step: 900 rmse: 0.01648146573819501
### iteration step: 950 rmse: 0.016447171683479155


In [4]:
pred_matrix = np.dot(P,Q.T)
print("예측 행렬:\n", np.round(pred_matrix, 3))

예측 행렬:
 [[3.991 0.897 1.306 2.002 1.663]
 [6.696 4.978 0.979 2.981 1.003]
 [6.677 0.391 2.987 3.977 3.986]
 [4.968 2.005 1.006 2.017 1.14 ]]
